In [18]:
# Instalar a biblioteca xlrd necessária para ler arquivos .xls
import sys
try:
    import xlrd
    print(" xlrd já está instalado")
except ImportError:
    print("Instalando xlrd...")
    %pip install xlrd
    import xlrd
    print(" xlrd instalado com sucesso!")


 xlrd já está instalado


In [19]:
import pandas as pd
import os

# Lista com os nomes dos 7 arquivos CSV na pasta dados
arquivos_csv = [
    'dados/PNAD_COVID_052020.csv',
    'dados/PNAD_COVID_062020.csv',
    'dados/PNAD_COVID_072020.csv',
    'dados/PNAD_COVID_082020.csv',
    'dados/PNAD_COVID_092020.csv',
    'dados/PNAD_COVID_102020.csv',
    'dados/PNAD_COVID_112020.csv'
]

# Dicionário para armazenar os DataFrames carregados
dfs = {}

# Carregar cada arquivo CSV
for arquivo in arquivos_csv:
    if os.path.exists(arquivo):
        print(f"\n{'='*60}")
        print(f"Carregando: {arquivo}")
        print(f"{'='*60}")
        try:
            # Carregar o arquivo CSV
            df = pd.read_csv(arquivo, low_memory=False)
            dfs[arquivo] = df
            print(f"Carregado com sucesso!")
            print(f"  Dimensoes: {df.shape[0]} linhas x {df.shape[1]} colunas")
        except Exception as e:
            print(f"Erro ao carregar: {e}")
    else:
        print(f"Arquivo nao encontrado: {arquivo}")

print(f"\n{'='*60}")
print(f"Total de arquivos carregados: {len(dfs)}")
print(f"{'='*60}")



Carregando: dados/PNAD_COVID_052020.csv
Carregado com sucesso!
  Dimensoes: 349306 linhas x 114 colunas

Carregando: dados/PNAD_COVID_062020.csv
Carregado com sucesso!
  Dimensoes: 381270 linhas x 114 colunas

Carregando: dados/PNAD_COVID_072020.csv
Carregado com sucesso!
  Dimensoes: 384166 linhas x 145 colunas

Carregando: dados/PNAD_COVID_082020.csv
Carregado com sucesso!
  Dimensoes: 386520 linhas x 145 colunas

Carregando: dados/PNAD_COVID_092020.csv
Carregado com sucesso!
  Dimensoes: 387298 linhas x 145 colunas

Carregando: dados/PNAD_COVID_102020.csv
Carregado com sucesso!
  Dimensoes: 380461 linhas x 145 colunas

Carregando: dados/PNAD_COVID_112020.csv
Carregado com sucesso!
  Dimensoes: 381438 linhas x 148 colunas

Total de arquivos carregados: 7


In [20]:
# # Mostrar as primeiras linhas de cada arquivo carregado
# for nome_arquivo, df in dfs.items():
#     print(f"\n{'='*80}")
#     print(f"Primeiras linhas de: {nome_arquivo}")
#     print(f"{'='*80}")
#     print(f"\nShape: {df.shape[0]} linhas x {df.shape[1]} colunas")
#     print(f"\nColunas: {list(df.columns)}")
#     print(f"\nPrimeiras 10 linhas:")
#     display(df.head(10))
#     print("\n")


In [21]:
# Transformar o arquivo PNAD_COVID_072020.csv
# 1. Transformar coluna Ano em colunas dummy (uma coluna para cada ano)
# 2. Transformar apenas as colunas específicas listadas em codigos.ipynb

# Carregar o arquivo específico
df_072020 = dfs['dados/PNAD_COVID_072020.csv'].copy()

print("Shape original:", df_072020.shape)
print("\nColunas originais:", list(df_072020.columns)[:10], "...")

# Lista de colunas específicas a serem transformadas (do arquivo codigos.ipynb)
colunas_especificas = [
    'A005',
    'B0011', 'B0012', 'B0013', 'B0014', 'B0015', 'B0016', 'B0017', 'B0018', 'B0019',
    'B00110', 'B00111', 'B00112', 'B00113',
    'B002', 'B0031',
    'B0041', 'B0042', 'B0043', 'B0044', 'B0045', 'B0046',
    'B005', 'B006', 'B008',
    'B0101', 'B0102', 'B0103', 'B0104', 'B0105', 'B0106',
    'B011',
    'C001', 'C01012', 'C01022', 'C011A12', 'C011A22', 'C012', 'C013',
    'D0013', 'D0023', 'D0033', 'D0043', 'D0053', 'D0063',
    'E001',
    'F001', 'F0021', 'F002A1', 'F002A2', 'F002A3', 'F002A4', 'F002A5'
]

# Filtrar apenas as colunas que existem no DataFrame
colunas_vdf = [col for col in colunas_especificas if col in df_072020.columns]
colunas_nao_encontradas = [col for col in colunas_especificas if col not in df_072020.columns]

# Identificar todas as colunas que começam com V, D, F, A, C, B, E (para removê-las)
import re
# Padrão para colunas que começam com V, D, F, A, C, B, E seguido de números/letras
# Exclui palavras completas como 'Ano' e 'CAPITAL'
padrao_vdf = re.compile(r'^(V|D|F|A|C|B|E)[\dA-Z]')

# Todas as colunas V, D, F, A, C, B, E no DataFrame (excluindo 'Ano' e 'CAPITAL')
todas_colunas_vdf = [col for col in df_072020.columns 
                     if padrao_vdf.match(col) and col not in ['Ano', 'CAPITAL']]

# Colunas V, D, F, A, C, B, E que NÃO estão na lista específica (devem ser removidas)
colunas_para_remover = [col for col in todas_colunas_vdf if col not in colunas_vdf]

# Colunas de identificação (não começam com V, D, F, A, C, B, E seguido de números/letras)
# Inclui 'Ano' e 'CAPITAL' explicitamente
colunas_outras = [col for col in df_072020.columns 
                  if (not padrao_vdf.match(col) or col in ['Ano', 'CAPITAL']) 
                  and col not in colunas_vdf]

print(f"\nColunas específicas a transformar: {len(colunas_vdf)}")
print(f"Exemplos: {colunas_vdf[:5]}")
if colunas_nao_encontradas:
    print(f"\nAtenção: {len(colunas_nao_encontradas)} colunas não encontradas no DataFrame:")
    print(f"Exemplos: {colunas_nao_encontradas[:5]}")
print(f"\nColunas V/D/F/A/C/B/E a serem removidas (não estão na lista): {len(colunas_para_remover)}")
print(f"Exemplos: {colunas_para_remover[:5]}")
print(f"\nColunas de identificação (mantidas): {len(colunas_outras)}")
print(f"Exemplos: {colunas_outras[:5]}")

# Remover as colunas V, D, F, A, C, B, E que não estão na lista específica
df_072020 = df_072020.drop(columns=colunas_para_remover)
print(f"\nApós remover colunas não desejadas: {df_072020.shape}")

# 1. Transformar coluna Ano em colunas dummy
anos_unicos = df_072020['Ano'].unique()
print(f"\nAnos únicos encontrados: {anos_unicos}")

# Criar colunas dummy para Ano
for ano in anos_unicos:
    df_072020[f'Ano_{int(ano)}'] = (df_072020['Ano'] == ano).astype(int)

# Remover a coluna Ano original
df_072020 = df_072020.drop(columns=['Ano'])

print(f"\nApós criar colunas dummy de Ano: {df_072020.shape}")

# 2. Transformar apenas as colunas específicas em linhas usando melt
# Manter as outras colunas como identificadores
# Filtrar apenas colunas que existem no DataFrame (remover 'Ano' que já foi removido)
colunas_id_vars = [col for col in colunas_outras if col in df_072020.columns]
id_vars = colunas_id_vars + [col for col in df_072020.columns if col.startswith('Ano_')]

# Garantir que todas as colunas em colunas_vdf existem no DataFrame
colunas_vdf_final = [col for col in colunas_vdf if col in df_072020.columns]

print(f"\nFazendo melt das {len(colunas_vdf_final)} colunas específicas...")
df_transformado = pd.melt(
    df_072020,
    id_vars=id_vars,
    value_vars=colunas_vdf_final,
    var_name='Variavel',
    value_name='Valor'
)

print(f"\nShape após transformação: {df_transformado.shape}")
print(f"\nPrimeiras linhas do resultado:")
display(df_transformado.head(10))


Shape original: (384166, 145)

Colunas originais: ['Ano', 'UF', 'CAPITAL', 'RM_RIDE', 'V1008', 'V1012', 'V1013', 'V1016', 'Estrato', 'UPA'] ...

Colunas específicas a transformar: 53
Exemplos: ['A005', 'B0011', 'B0012', 'B0013', 'B0014']

Colunas V/D/F/A/C/B/E a serem removidas (não estão na lista): 85
Exemplos: ['V1008', 'V1012', 'V1013', 'V1016', 'V1022']

Colunas de identificação (mantidas): 7
Exemplos: ['Ano', 'UF', 'CAPITAL', 'RM_RIDE', 'Estrato']

Após remover colunas não desejadas: (384166, 60)

Anos únicos encontrados: [2020]

Após criar colunas dummy de Ano: (384166, 60)

Fazendo melt das 53 colunas específicas...

Shape após transformação: (20360798, 9)

Primeiras linhas do resultado:


,UF,CAPITAL,RM_RIDE,Estrato,UPA,posest,Ano_2020,Variavel,Valor
0,11,11.0,NaN,1110011,110015970,1114,1,A005,5.0
1,11,11.0,NaN,1110011,110015970,1123,1,A005,7.0
2,11,11.0,NaN,1110011,110015970,1112,1,A005,2.0
3,11,11.0,NaN,1110011,110015970,1112,1,A005,2.0
4,11,11.0,NaN,1110011,110015970,1126,1,A005,2.0
5,11,11.0,NaN,1110011,110015970,1115,1,A005,2.0
6,11,11.0,NaN,1110011,110015970,1122,1,A005,2.0
7,11,11.0,NaN,1110011,110015970,1112,1,A005,2.0
8,11,11.0,NaN,1110011,110015970,1121,1,A005,2.0
9,11,11.0,NaN,1110011,110015970,1112,1,A005,2.0


In [22]:
# Salvar o DataFrame transformado em CSV
nome_arquivo_saida = 'PNAD_COVID_072020_transformado.csv'

print(f"Salvando arquivo: {nome_arquivo_saida}")
print(f"Shape do DataFrame: {df_transformado.shape}")

try:
    df_transformado.to_csv(nome_arquivo_saida, index=False, encoding='utf-8')
    print(f"\nArquivo salvo com sucesso: {nome_arquivo_saida}")
    print(f"Tamanho aproximado: {os.path.getsize(nome_arquivo_saida) / (1024*1024):.2f} MB")
except Exception as e:
    print(f"\nErro ao salvar arquivo: {e}")


Salvando arquivo: PNAD_COVID_072020_transformado.csv
Shape do DataFrame: (20360798, 9)

Arquivo salvo com sucesso: PNAD_COVID_072020_transformado.csv
Tamanho aproximado: 819.46 MB
